# IMDb Movie Review Analysis with PySpark
**Module 1 – Homework 1.2** · Shashank Reddy Reddimalla (1RV23CD050)

Dataset: [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) — columns `review`, `sentiment`.

Runs as-is in **Google Colab** (Java is preinstalled). Steps: Load → Explore → Clean → Analyze → Transform → Save.

In [ ]:
# Install PySpark (needed once per Colab session; skip if already installed locally)
!pip install -q "pyspark>=3.5,<4.0"

Upload **`IMDB Dataset.csv`** (unzip the Kaggle download first). In Colab, run the cell below and pick the file; locally, put it in a `data/` folder next to this notebook.

In [ ]:
import os
DATA_PATH = "data/IMDB Dataset.csv"
try:
    from google.colab import files  # only available in Colab
    if not os.path.exists(DATA_PATH):
        os.makedirs("data", exist_ok=True)
        uploaded = files.upload()
        name = next(iter(uploaded))
        os.replace(name, DATA_PATH)
except ImportError:
    pass
print("Dataset found:", os.path.exists(DATA_PATH))

## 0. Start a Spark session

In [ ]:
import sys
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

spark = (SparkSession.builder.appName("IMDbMovieReviewAnalysis")
         .master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

## 1. Load the dataset
Reviews contain commas, quotes and line breaks, so `multiLine` and `escape` are required for correct parsing.

In [ ]:
schema = StructType([StructField("review", StringType(), True),
                     StructField("sentiment", StringType(), True)])
raw_df = (spark.read.option("header", True).option("multiLine", True)
          .option("quote", '"').option("escape", '"')
          .schema(schema).csv(DATA_PATH))
raw_count = raw_df.count()
print(f"Rows loaded: {raw_count:,}")

## 2. Explore the data

In [ ]:
raw_df.printSchema()

In [ ]:
raw_df.show(5, truncate=80)

## 3. Clean the data – remove null or empty reviews

In [ ]:
null_reviews  = raw_df.filter(F.col("review").isNull()).count()
empty_reviews = raw_df.filter(F.col("review").isNotNull() & (F.trim("review") == "")).count()

clean_df = (raw_df.filter(F.col("review").isNotNull() & (F.trim(F.col("review")) != ""))
                  .withColumn("sentiment", F.lower(F.trim("sentiment")))
                  .cache())
clean_count = clean_df.count()
print(f"Null reviews: {null_reviews:,} | Empty reviews: {empty_reviews:,}")
print(f"Rows after cleaning: {clean_count:,} (removed {raw_count - clean_count:,})")

## 4. Analyze the data

In [ ]:
print(f"Total number of reviews: {clean_count:,}")
sentiment_counts = (clean_df.groupBy("sentiment").agg(F.count("*").alias("num_reviews"))
                    .withColumn("percent", F.round(F.col("num_reviews") * 100.0 / clean_count, 2))
                    .orderBy(F.desc("num_reviews")))
sentiment_counts.show()

## 5. Transform the data
Add `review_length` (number of characters), then keep reviews longer than 500 characters.

In [ ]:
transformed_df = clean_df.withColumn("review_length", F.length("review"))
transformed_df.select("review", "sentiment", "review_length").show(5, truncate=60)

In [ ]:
long_reviews_df = transformed_df.filter(F.col("review_length") > 500)
long_count = long_reviews_df.count()
print(f"Reviews longer than 500 characters: {long_count:,} ({long_count * 100 / clean_count:.1f}%)")
long_reviews_df.select("review", "sentiment", "review_length").show(10, truncate=60)

**Extra insight:** are positive or negative reviews longer?

In [ ]:
(transformed_df.groupBy("sentiment")
   .agg(F.round(F.avg("review_length"), 1).alias("avg_length"),
        F.min("review_length").alias("min_length"),
        F.max("review_length").alias("max_length"),
        F.sum(F.when(F.col("review_length") > 500, 1).otherwise(0)).alias("reviews_over_500"))
   .orderBy("sentiment").show())

## 6. Save the cleaned and transformed DataFrame as a new CSV

In [ ]:
OUTPUT_DIR = "output/cleaned_reviews"
try:
    (transformed_df.coalesce(1).write.mode("overwrite")
        .option("header", True).option("quote", '"').option("escape", '"').csv(OUTPUT_DIR))
    print("Saved to folder:", OUTPUT_DIR, os.listdir(OUTPUT_DIR))
except Exception as e:   # e.g. Windows without winutils.exe
    print("Spark writer failed, saving with pandas:", type(e).__name__)
    os.makedirs("output", exist_ok=True)
    transformed_df.toPandas().to_csv(OUTPUT_DIR + ".csv", index=False)
    print("Saved to:", OUTPUT_DIR + ".csv")

In [ ]:
# Read the saved file back to confirm it was written correctly
check = (spark.read.option("header", True).option("multiLine", True)
         .option("quote", '"').option("escape", '"').csv(OUTPUT_DIR if os.path.isdir(OUTPUT_DIR) else OUTPUT_DIR + ".csv"))
print("Rows in saved CSV:", check.count(), "| columns:", check.columns)

In [ ]:
spark.stop()